# 07 · Visual showcase

**Purpose.** Presentation-grade versions of the project's key findings, so the team can pick the
strongest visuals for the final deck. This notebook adds **no new analysis**: every number is either
recomputed from `data/processed/` with the same specifications as notebooks 01-05, or read from
`results/*.csv`, and the key values are asserted against those CSVs below.

**Output.** `figures/showcase/NN_name.png` (3200 x 1800 px, 16:9, 200 dpi) and `.svg`, light theme for
slides; a matching dark-theme PNG of each chart goes to `figures/showcase/dark/`.

**Design rules used throughout.**
* One headline sentence per chart (the takeaway), one subtitle line saying what is plotted, a source line.
* Two story colours only: **blue** = students / the corrected estimate, **orange** = the zero-grade cases
  (apparent dropouts). Grey = the uncorrected / "all students" estimate. Validated for colour-vision
  deficiency (blue/orange, CVD delta-E 24.7 light, 26.8 dark).
* Direct labels instead of legends; no dual axes; zoomed axes are only used for dot plots and are stated.
* Status colours (pass / watch / fail) appear only in the diagnostics scorecard and always with a word.

In [1]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
PROC, RES = Path("../data/processed"), Path("../results")
OUT = Path("../figures/showcase")
(OUT / "dark").mkdir(parents=True, exist_ok=True)

por = pd.read_csv(PROC / "student-por_clean.csv")
mat = pd.read_csv(PROC / "student-mat_clean.csv")
for d in (por, mat):
    d["zero_grade"] = d["zero_grade"].astype(bool)
nz, mnz = por[~por.zero_grade], mat[~mat.zero_grade]

hyp = pd.read_csv(RES / "hypothesis_tests.csv")
reg = pd.read_csv(RES / "regression_comparison.csv")
diag = pd.read_csv(RES / "diagnostics_scorecard.csv", keep_default_na=False, na_values={"p_value": [""], "statistic": [""]})
ba = pd.read_csv(RES / "zero_grade_before_after.csv").set_index("sample")
print(por.shape, mat.shape, "zero-grade cases:", por.zero_grade.sum(), mat.zero_grade.sum())

(649, 35) (395, 35) zero-grade cases: 15 38


## Theme: light (slides) and dark, same hues stepped for each surface

In [2]:
LIGHT = dict(name="light", bg="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
             axis="#c3c2b7", blue="#2a78d6", blue_soft="#b7d3f6", orange="#eb6834", grey="#9a9890",
             band="#e8e7e1")
DARK = dict(name="dark", bg="#1a1a19", ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a",
            axis="#383835", blue="#3987e5", blue_soft="#1c3d66", orange="#d95926", grey="#8a8982",
            band="#2c2c2a")
STATUS = {"green": ("#0ca30c", "Pass", "#ffffff"), "amber": ("#fab219", "Watch", "#0b0b0b"),
          "red": ("#d03b3b", "Fail", "#ffffff"), "n/a": (None, "n/a", None)}
FONT = ["Helvetica Neue", "Arial", "DejaVu Sans"]
SRC = "Data: UCI Student Performance (Cortez 2008), Portuguese course, n = 649."


def style(T):
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.sans-serif": FONT, "font.size": 18,
        "figure.facecolor": T["bg"], "axes.facecolor": T["bg"], "savefig.facecolor": T["bg"],
        "text.color": T["ink"], "axes.labelcolor": T["ink2"], "axes.labelsize": 18,
        "xtick.color": T["ink2"], "ytick.color": T["ink2"], "xtick.labelsize": 17, "ytick.labelsize": 17,
        "axes.edgecolor": T["axis"], "axes.linewidth": 1.2, "axes.grid": False, "grid.color": T["grid"],
        "grid.linewidth": 1.0, "axes.spines.top": False, "axes.spines.right": False,
        "xtick.major.size": 0, "ytick.major.size": 0, "xtick.major.pad": 8, "ytick.major.pad": 8,
        "svg.fonttype": "path", "axes.unicode_minus": True, "legend.frameon": False,
    })


def m(x, nd=2, sign=True):
    """Number with a real minus sign (U+2212)."""
    s = f"{x:+.{nd}f}" if sign else f"{x:.{nd}f}"
    return s.replace("-", "−")


RA, LA = r"$\rightarrow$", r"$\leftarrow$"   # Helvetica Neue has no arrow glyphs; mathtext draws them
SUP = dict(zip("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹"))


def sci(x):
    mant, ex = f"{x:.1e}".split("e")
    return f"{mant}×10" + "".join(SUP[ch] for ch in str(int(ex)))


def fmt_p(p):
    if p < 0.001:
        return f"p = {sci(p)}"
    return f"p = {p:.3f}" if p < 0.1 else f"p = {p:.2f}"


def halo(T, w=4):
    from matplotlib import patheffects as pe
    return [pe.withStroke(linewidth=w, foreground=T["bg"])]


def canvas(T, title, sub, src=SRC, w=16, h=9):
    style(T)
    fig = plt.figure(figsize=(w, h))
    fig.text(0.045, 0.94, title, fontsize=30, weight="bold", color=T["ink"], va="top")
    fig.text(0.045, 0.865, sub, fontsize=18, color=T["ink2"], va="top")
    if src:
        fig.text(0.045, 0.028, src, fontsize=13, color=T["muted"], va="bottom")
    return fig


def ygrid(ax, T):
    ax.yaxis.grid(True, color=T["grid"], lw=1); ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)


def xgrid(ax, T):
    ax.xaxis.grid(True, color=T["grid"], lw=1); ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)


SAVED = []


def save(fig, name, T):
    if T["name"] == "light":
        fig.savefig(OUT / f"{name}.png", dpi=200)
        fig.savefig(OUT / f"{name}.svg")
        SAVED.append(name)
    else:
        fig.savefig(OUT / "dark" / f"{name}.png", dpi=200)
    plt.close(fig)


def jitter(n, w, seed):
    return np.random.default_rng(seed).uniform(-w, w, n)

## Numbers used in the charts (recomputed, then checked against results/)

In [3]:
ST = {1: "<2 h", 2: "2-5 h", 3: "5-10 h", 4: ">10 h"}
grp = por.groupby("studytime").G3.agg(["mean", "std", "count"])
grp["half"] = stats.t.ppf(0.975, grp["count"] - 1) * grp["std"] / np.sqrt(grp["count"])
tuk = hyp[(hyp.test.str.startswith("Tukey")) & (hyp["sample"] == "all")].set_index("test")
t13, t34 = tuk.loc["Tukey HSD <2 h vs 5-10 h"], tuk.loc["Tukey HSD 5-10 h vs >10 h"]
anova = hyp[(hyp.test == "One-way ANOVA G3 ~ studytime") & (hyp["sample"] == "all")].iloc[0]

lad = reg[(reg.section == "ladder (numeric studytime)") & (reg.term == "studytime")]
lad_all, lad_nz = lad[lad["sample"] == "all"].reset_index(drop=True), lad[lad["sample"] == "G3>0"].reset_index(drop=True)

assert np.allclose(grp["mean"].round(2).values, [10.84, 12.09, 13.23, 13.06])
assert np.allclose(lad_all.estimate.round(2).values, [0.97, 0.45, 0.42, 0.05])
print(grp.round(2)); print(lad_all[["model", "estimate", "ci_lo", "ci_hi", "r2"]].round(3))

            mean   std  count  half
studytime                          
1          10.84  3.22    212  0.44
2          12.09  3.24    305  0.37
3          13.23  2.50     97  0.50
4          13.06  3.04     35  1.04
                       model  estimate  ci_lo  ci_hi     r2
0              M1 study time     0.973  0.691  1.255  0.062
1  M2 + academic/demographic     0.447  0.195  0.698  0.287
2         M3 + family/social     0.415  0.167  0.664  0.338
3                M4 + G1, G2     0.053 -0.049  0.156  0.856


## 01 · Effort pays, then plateaus

In [4]:
def chart01(T):
    fig = canvas(T, "Studying more pays off up to 5-10 hours a week, then it plateaus",
                 "Mean final grade G3 (0-20) by self-reported weekly study time, with 95% confidence intervals",
                 SRC + "  Axis zoomed to 9-15 to show the gaps; ANOVA F(3, 645) = "
                 f"{anova.value:.1f}, p < 0.001, eta² = {anova.effect:.2f}. Gaps: Tukey HSD.")
    ax = fig.add_axes([0.10, 0.17, 0.86, 0.62])
    x = np.arange(4)
    mu, hf = grp["mean"].values, grp["half"].values
    ax.plot(x, mu, color=T["muted"], lw=2, zorder=1)
    ax.vlines(x, mu - hf, mu + hf, color=T["blue"], lw=4, zorder=2)
    ax.scatter(x, mu, s=420, color=T["blue"], edgecolor=T["bg"], lw=3, zorder=3)
    for xi, (v, h) in enumerate(zip(mu, hf)):
        dy = {0: -0.32, 1: -0.32, 2: 0.32, 3: 0.0}[xi]   # keep labels off the connecting line
        ax.text(xi + 0.1, v + dy, f"{v:.1f}", fontsize=26, weight="bold", color=T["ink"], va="center")
    # gap bracket <2 h -> 5-10 h
    yb = 14.55
    ax.plot([0, 0, 2, 2], [yb - 0.15, yb, yb, yb - 0.15], color=T["ink2"], lw=1.6)
    ax.text(1, yb + 0.1, f"+{t13.value:.2f} points", ha="center", va="bottom", fontsize=24, weight="bold")
    ax.text(1, yb - 0.12, f"95% CI {t13.effect_ci_lo:.2f} to {t13.effect_ci_hi:.2f}", ha="center", va="top",
            fontsize=16, color=T["ink2"])
    # plateau bracket (below the >10 h interval, which ends at ~12.0)
    yp = 11.7
    ax.plot([2, 2, 3, 3], [yp + 0.15, yp, yp, yp + 0.15], color=T["ink2"], lw=1.6)
    ax.text(2.5, yp - 0.1, f"Plateau: {m(t34.value)}, p = {t34.p_value:.2f}", ha="center", va="top",
            fontsize=17, color=T["ink2"])
    ax.set_xticks(x, [f"{ST[i]}\nn = {int(n)}" for i, n in zip(grp.index, grp["count"])], fontsize=19)
    ax.set_xlim(-0.45, 3.55); ax.set_ylim(9, 15.4)
    ax.set_ylabel("Mean final grade G3")
    ygrid(ax, T)
    return fig


for T in (LIGHT, DARK):
    save(chart01(T), "01_effort_pays_then_plateaus", T)

## 02 · The coefficient ladder melts as controls are added

In [5]:
RUNG_LABELS = ["M1\nstudy time only", "M2\n+ failures, absences,\nschool, ambition", "M3\n+ family & social\nfactors",
               "M4\n+ earlier grades\nG1, G2"]


def chart02(T):
    fig = canvas(T, "Compare like with like and the study-time effect melts away",
                 "Study-time coefficient (G3 points per study-time category), all 649 students, OLS with HC3 95% CI",
                 SRC + f"  Without the 15 zero-grade cases: {', '.join(m(v) for v in lad_nz.estimate)}.")
    ax = fig.add_axes([0.10, 0.36, 0.86, 0.44])
    axr = fig.add_axes([0.10, 0.215, 0.86, 0.10], sharex=ax)
    x = np.arange(4)
    b, lo, hi, r2 = lad_all.estimate.values, lad_all.ci_lo.values, lad_all.ci_hi.values, lad_all.r2.values
    for xi in x[1:]:   # dashed outline of the M1 bar = the part that "melted"
        ax.add_patch(Rectangle((xi - 0.28, 0), 0.56, b[0], fill=False, ls=(0, (4, 3)), lw=1.6,
                               ec=T["muted"], zorder=1))
    ax.bar(x, b, width=0.56, color=T["blue"], zorder=2)
    ax.vlines(x, lo, hi, color=T["ink"], lw=2, zorder=3)
    ax.axhline(0, color=T["axis"], lw=1.4)
    for xi in x:
        ax.text(xi + 0.32, b[xi], m(b[xi]), fontsize=26, weight="bold", va="center", ha="left")
        if xi:
            ax.text(xi, b[0] + 0.05, f"{b[xi] / b[0]:.0%} of M1 left", ha="center", va="bottom", fontsize=16,
                    color=T["ink2"])
    ax.text(0, b[0] + 0.05, " ", fontsize=16)
    ax.set_ylim(-0.12, 1.42); ax.set_yticks([0, 0.5, 1.0], ["0", "+0.5", "+1.0"])
    ax.set_ylabel("Study-time effect")
    ygrid(ax, T)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="x", labelbottom=False)
    # R2 strip
    axr.barh([0] * 4, [0] * 4)  # keep sharex scaling stable
    axr.cla()
    for xi in x:
        axr.add_patch(Rectangle((xi - 0.28, 0), 0.56, 1, color=T["band"], lw=0))
        axr.add_patch(Rectangle((xi - 0.28, 0), 0.56 * r2[xi], 1, color=T["grey"], lw=0))
        axr.text(xi + 0.32, 0.5, f"R² {r2[xi]:.2f}", va="center", fontsize=17, color=T["ink2"])
    axr.set_ylim(0, 1); axr.set_yticks([])
    for s in ["left", "bottom"]:
        axr.spines[s].set_visible(False)
    axr.set_xticks(x, RUNG_LABELS, fontsize=16)
    axr.set_xlim(-0.5, 3.75)
    axr.text(-0.5, 1.22, "Share of grade variation explained (bar = R², full width = 100%)", fontsize=15,
             color=T["ink2"], transform=axr.transData)
    return fig


for T in (LIGHT, DARK):
    save(chart02(T), "02_coefficient_ladder_melts", T)

## 03 · The spike at zero

In [6]:
def chart03(T):
    cnt = por.G3.value_counts().reindex(range(21), fill_value=0)
    fig = canvas(T, "Fifteen students finished the year with a final grade of zero",
                 "Number of students at each final grade G3 (0-20), Portuguese course")
    ax = fig.add_axes([0.08, 0.15, 0.88, 0.64])
    cols = [T["orange"] if g == 0 else T["blue"] for g in cnt.index]
    ax.bar(cnt.index, cnt.values, width=0.86, color=cols, zorder=2)
    ax.set_xticks(range(0, 21, 2)); ax.set_xlim(-0.8, 20.8)
    ax.set_xlabel("Final grade G3"); ax.set_ylabel("Students")
    ygrid(ax, T)
    ax.annotate(f"{cnt[0]} students scored 0.\nAll {cnt[0]} had zero recorded absences.\n"
                f"(Only {int((nz.absences == 0).sum())} of the other {len(nz)} did.)",
                xy=(0, cnt[0] + 1.5), xytext=(1.0, 72), fontsize=19, color=T["ink"], va="bottom",
                arrowprops=dict(arrowstyle="-|>", color=T["ink2"], lw=1.6, shrinkB=2,
                                connectionstyle="arc3,rad=0.25"))
    peak = cnt.idxmax()
    ax.text(peak, cnt[peak] + 1.5, f"{cnt[peak]}", ha="center", va="bottom", fontsize=16, color=T["ink2"])
    return fig


for T in (LIGHT, DARK):
    save(chart03(T), "03_spike_at_zero", T)

## 04 · The Perfect Attendance Paradox (Portuguese)

In [7]:
def simple_fit(d):
    f = smf.ols("G3 ~ log_abs", d).fit()
    return f.params["Intercept"], f.params["log_abs"]


assert np.isclose(simple_fit(por)[1], ba.loc["Portuguese, all", "logabs_simple"])
assert np.isclose(simple_fit(nz)[1], ba.loc["Portuguese, G3 > 0", "logabs_simple"])


def paradox_scatter(ax, d, T, seed, label_lines=True, lbl_fs=16):
    z, o = d[d.zero_grade], d[~d.zero_grade]
    ax.scatter(o.absences + jitter(len(o), 0.3, seed), o.G3 + jitter(len(o), 0.3, seed + 1), s=34,
               color=T["blue"], alpha=0.35, lw=0, zorder=2)
    ax.scatter(z.absences + jitter(len(z), 0.45, seed + 2), z.G3 + jitter(len(z), 0.45, seed + 3), s=110,
               color=T["orange"], edgecolor=T["bg"], lw=1.5, zorder=4)
    xs = np.linspace(0, d.absences.max(), 200)
    a1, b1 = simple_fit(d); a2, b2 = simple_fit(o)
    ax.plot(xs, a1 + b1 * np.log1p(xs), color=T["grey"], lw=3.5, ls=(0, (6, 3)), zorder=5)
    ax.plot(xs, a2 + b2 * np.log1p(xs), color=T["blue"], lw=3.5, zorder=5)
    ax.set_ylim(-1.2, 20.5); ax.set_xlim(-1.2, d.absences.max() + 1)
    ax.set_yticks([0, 5, 10, 15, 20])
    ygrid(ax, T)
    return (a1, b1), (a2, b2), xs


def chart04(T):
    z = por[por.zero_grade]
    fig = canvas(T, "The Perfect Attendance Paradox: every zero grade had zero absences",
                 "Left: each dot is a student (jittered). Right: share of students with no recorded absences",
                 SRC + "  Fitted lines: OLS of G3 on log(absences + 1).")
    ax = fig.add_axes([0.075, 0.15, 0.54, 0.65])
    (a1, b1), (a2, b2), xs = paradox_scatter(ax, por, T, 7)
    ax.set_xlabel("Recorded absences"); ax.set_ylabel("Final grade G3")
    ax.annotate(f"{len(z)} zero-grade cases,\nall at 0 absences", xy=(0.6, 0.3), xytext=(4.5, 2.2), fontsize=18,
                color=T["ink"], arrowprops=dict(arrowstyle="-|>", color=T["ink2"], lw=1.5))
    xe = xs[-1]
    ax.text(xe, a1 + b1 * np.log1p(xe) + 0.9, f"with them: {m(b1 * np.log(11), 1)} pts for 10 absences",
            ha="right", va="bottom", fontsize=17, color=T["ink2"], path_effects=halo(T))
    ax.text(xe, a2 + b2 * np.log1p(xe) - 0.9, f"without them: {m(b2 * np.log(11), 1)} pts",
            ha="right", va="top", fontsize=17, color=T["blue"], weight="bold", path_effects=halo(T))
    # right: share with zero absences
    axb = fig.add_axes([0.70, 0.27, 0.27, 0.42])
    shares = [(f"Zero-grade cases\n(n = {len(z)})", (z.absences == 0).mean(), T["orange"], int((z.absences == 0).sum()), len(z)),
              (f"Everyone else\n(n = {len(nz)})", (nz.absences == 0).mean(), T["blue"], int((nz.absences == 0).sum()), len(nz))]
    for i, (lab, s, c, k, n) in enumerate(shares):
        axb.barh(i, 1, color=T["band"], height=0.56, lw=0)
        axb.barh(i, s, color=c, height=0.56, lw=0)
        axb.text(s + 0.03 if s < 0.8 else s - 0.03, i, f"{s:.0%}", va="center", ha="left" if s < 0.8 else "right",
                 fontsize=28, weight="bold", color=T["ink"] if s < 0.8 else "#ffffff")
    axb.set_yticks([0, 1], [s[0] for s in shares], fontsize=17)
    axb.invert_yaxis(); axb.set_xlim(0, 1); axb.set_xticks([])
    for s in ["left", "bottom"]:
        axb.spines[s].set_visible(False)
    axb.set_title("No recorded absences", loc="left", fontsize=19, weight="bold", color=T["ink"], pad=14)
    p_rand = (por.absences == 0).mean() ** len(z)
    axb.text(0, 1.75, f"Chance of {len(z)} of {len(z)} if they were\na random draw of students: {sci(p_rand)}",
             fontsize=15, color=T["ink2"], transform=axb.transData)
    return fig


for T in (LIGHT, DARK):
    save(chart04(T), "04_perfect_attendance_paradox", T)

## 05 · Trajectories: the zero-grade cases were fading before G3

In [8]:
def chart05(T):
    z = por[por.zero_grade].sort_values(["G2", "G1"])
    fig = canvas(T, "The zero-grade students were already fading out before the final",
                 "Grades across the three periods. Orange: each of the 15 zero-grade cases. Blue: all other students")
    ax = fig.add_axes([0.08, 0.15, 0.53, 0.65])
    x = np.array([0, 1, 2])
    G = nz[["G1", "G2", "G3"]]
    q10, q25, q50, q75, q90 = (G.quantile(q).values for q in (0.10, 0.25, 0.5, 0.75, 0.90))
    ax.fill_between(x, q10, q90, color=T["blue_soft"], alpha=0.45, lw=0, zorder=1)
    ax.fill_between(x, q25, q75, color=T["blue_soft"], alpha=0.9, lw=0, zorder=1)
    ax.plot(x, q50, color=T["blue"], lw=4, marker="o", ms=11, zorder=3)
    off = np.linspace(-0.28, 0.28, len(z))
    for k, (_, r) in enumerate(z.iterrows()):
        ax.plot(x, np.array([r.G1, r.G2, r.G3]) + off[k], color=T["orange"], lw=2.2, alpha=0.85,
                marker="o", ms=6, mec=T["bg"], mew=1, zorder=4)
    ax.set_xticks(x, ["G1\n(period 1)", "G2\n(period 2)", "G3\n(final)"])
    ax.set_xlim(-0.15, 2.15); ax.set_ylim(-1, 20.5); ax.set_ylabel("Grade (0-20)")
    ax.set_yticks([0, 5, 10, 15, 20])
    ygrid(ax, T)
    ax.text(2.07, q50[2], f"Other {len(nz)}\nstudents:\nmedian {q50[2]:.0f}", va="center", fontsize=16,
            color=T["blue"], weight="bold", clip_on=False)
    ax.text(2.07, q90[2] + 1.6, "shading:\nmiddle 50%\nand 80%", fontsize=14, color=T["ink2"], va="center",
            clip_on=False)
    ax.text(2.07, 0, f"{len(z)} zero-\ngrade cases", fontsize=16, color=T["orange"], weight="bold", va="center",
            clip_on=False)
    # side panel: the facts
    zg = por[por.zero_grade]
    facts = [(f"{zg.G1.mean():.1f}", "mean G1 of the zero-grade cases"),
             (f"{zg.G2.mean():.1f}", "their mean G2: already falling"),
             (f"{int((zg.G2 == 0).sum())} of {len(zg)}", "already had G2 = 0"),
             (f"{int((nz.G2 == 0).sum())}", "other students with G2 = 0")]
    y0 = 0.78
    for big, small in facts:
        fig.text(0.735, y0, big, fontsize=40, weight="bold", color=T["orange"] if big != "0" else T["blue"], va="top")
        fig.text(0.735, y0 - 0.085, small, fontsize=17, color=T["ink2"], va="top")
        y0 -= 0.155
    fig.text(0.735, 0.15, "We read them as apparent dropouts;\nthe dataset does not say why.",
             fontsize=15, color=T["muted"], style="italic", va="top")
    return fig


for T in (LIGHT, DARK):
    save(chart05(T), "05_zero_grade_trajectories", T)

## 06 · Before/after: the absence effect with and without the zero-grade cases

In [9]:
ROWS6 = [("Portuguese", "M3", "controls model"), ("Portuguese", "M4", "+ earlier grades"),
         ("Maths", "M3", "controls model"), ("Maths", "M4", "+ earlier grades")]


def chart06(T):
    fig = canvas(T, "Drop the zero-grade cases and every “absences help” result vanishes",
                 "Coefficient on log(absences + 1), G3 points, with HC3 95% CI: all students vs zero-grade cases removed",
                 "Data: UCI Student Performance (Cortez 2008). Portuguese n = 649 (15 zero grades); "
                 "maths n = 395 (38 zero grades). Results from notebook 04.")
    ax = fig.add_axes([0.25, 0.15, 0.53, 0.62])
    for i, (subj, mod, desc) in enumerate(ROWS6):
        a, b = ba.loc[f"{subj}, all"], ba.loc[f"{subj}, G3 > 0"]
        y = i
        for r, c, dy, filled in [(a, T["grey"], -0.14, False), (b, T["blue"], 0.14, True)]:
            est, lo, hi, p = r[f"{mod}_logabs"], r[f"{mod}_logabs_lo"], r[f"{mod}_logabs_hi"], r[f"{mod}_logabs_p"]
            ax.hlines(y + dy, lo, hi, color=c, lw=3)
            ax.scatter(est, y + dy, s=260, color=c if filled else T["bg"], edgecolor=c, lw=3, zorder=3)
        ea, eb = a[f"{mod}_logabs"], b[f"{mod}_logabs"]
        ax.annotate("", xy=(eb, y + 0.14), xytext=(ea, y - 0.14),
                    arrowprops=dict(arrowstyle="-|>", color=T["ink2"], lw=1.4, shrinkA=9, shrinkB=9))
        fig.text(0.80, 0.735 - i * 0.155, f"{m(ea)}  {RA}  {m(eb)}", fontsize=22, weight="bold", va="center",
                 color=T["ink"])
        pa, pb = a[f"{mod}_logabs_p"], b[f"{mod}_logabs_p"]
        fig.text(0.80, 0.735 - i * 0.155 - 0.045, f"{fmt_p(pa)}  {RA}  {fmt_p(pb)}",
                 fontsize=14, color=T["ink2"], va="center")
    ax.axvline(0, color=T["ink2"], lw=1.4)
    ax.set_yticks(range(4), [f"{s} · {mo}\n{d}" for s, mo, d in ROWS6], fontsize=17)
    ax.set_ylim(3.6, -0.75); ax.set_xlim(-1.05, 1.55)
    ax.set_xticks([-1, -0.5, 0, 0.5, 1, 1.5], ["−1.0", "−0.5", "0", "+0.5", "+1.0", "+1.5"])
    xgrid(ax, T); ax.spines["left"].set_visible(False)
    ax.text(-0.04, -0.62, f"{LA} more absences, lower grade", ha="right", fontsize=15, color=T["ink2"])
    ax.text(0.04, -0.62, f"more absences, higher grade?! {RA}", ha="left", fontsize=15, color=T["ink2"])
    # direct series labels
    fig.text(0.80, 0.79, f"all  {RA}  zeros removed", fontsize=15, color=T["ink2"])
    ax.scatter([], [], s=150, color=T["bg"], edgecolor=T["grey"], lw=3, label="all students")
    ax.scatter([], [], s=150, color=T["blue"], label="zero-grade cases removed")
    ax.legend(loc="lower right", fontsize=15, handletextpad=0.3, borderaxespad=0.2)
    return fig


for T in (LIGHT, DARK):
    save(chart06(T), "06_absence_effect_before_after", T)

## 07 · Maths replication: small multiples

In [10]:
assert np.isclose(simple_fit(mat)[1], ba.loc["Maths, all", "logabs_simple"])


def chart07(T):
    fig = canvas(T, "Same trap in maths: remove the 38 zeros and the fitted line flips",
                 "Final grade vs recorded absences; grey dashed = fit on all students, blue = zero-grade cases removed",
                 "Data: UCI Student Performance (Cortez 2008). Fits: OLS of G3 on log(absences + 1); points jittered. "
                 "Labels: implied change from 0 to 10 absences.")
    for k, (d, name, x0) in enumerate([(por, "Portuguese", 0.075), (mat, "Maths", 0.545)]):
        ax = fig.add_axes([x0, 0.15, 0.415, 0.6])
        (a1, b1), (a2, b2), xs = paradox_scatter(ax, d, T, 11 + k)
        z = d[d.zero_grade]
        ax.set_title(f"{name}: n = {len(d)}, {len(z)} zero grades, all {int((z.absences == 0).sum())} at 0 absences",
                     loc="left", fontsize=18, weight="bold", color=T["ink"], pad=12)
        ax.set_xlabel("Recorded absences")
        if k == 0:
            ax.set_ylabel("Final grade G3")
        else:
            ax.tick_params(labelleft=False)
        xe = xs[-1]
        ya, yb = a1 + b1 * np.log1p(xe), a2 + b2 * np.log1p(xe)
        up = ya > yb
        ax.text(xe, ya + (0.8 if up else -0.8), f"all: {m(b1 * np.log(11), 1)} pts", ha="right",
                va="bottom" if up else "top", fontsize=17, color=T["ink2"])
        ax.text(xe, yb + (-0.8 if up else 0.8), f"zeros removed: {m(b2 * np.log(11), 1)} pts", ha="right",
                va="top" if up else "bottom", fontsize=17, color=T["blue"], weight="bold")
    return fig


for T in (LIGHT, DARK):
    save(chart07(T), "07_maths_replication", T)

## 08 · What predicts grades, in comparable units (model M3)

In [11]:
M3F = reg[reg.section == "M3 full coefficients"]
PD = reg[reg.section == "standardised / predicted difference"]


def pick(src, term):
    out = {}
    for s in ("all", "G3>0"):
        r = src[(src.term == term) & (src["sample"] == s)].iloc[0]
        out[s] = (r.estimate, r.ci_lo, r.ci_hi, r.p_value)
    return out


EFFECTS = [("Wants higher education", "yes vs no", pick(M3F, "C(higher)[T.yes]")),
           ("Study time", "<2 h vs 5-10 h a week", pick(PD, "studytime: study time <2 h -> 5-10 h")),
           ("Family support", "yes vs no", pick(M3F, "C(famsup)[T.yes]")),
           ("Absences", "10 vs none", pick(PD, "log_abs: absences 0 -> 10")),
           ("School MS", "vs school GP", pick(M3F, "C(school)[T.MS]")),
           ("Extra school support", "yes vs no (selection: weaker students get it)", pick(M3F, "C(schoolsup)[T.yes]")),
           ("One past failure", "vs none", pick(PD, "failures: failures 0 -> 1"))]
assert np.isclose(EFFECTS[0][2]["all"][0], 1.807, atol=1e-3) and np.isclose(EFFECTS[6][2]["all"][0], -1.421, atol=1e-3)


def chart08(T):
    fig = canvas(T, "Ambition and past failures outweigh an extra study bracket",
                 "Predicted difference in final grade (G3 points) for each contrast, everything else in model M3 held fixed",
                 SRC + "  OLS with HC3 95% CI. Associations, not causes.")
    ax = fig.add_axes([0.33, 0.17, 0.45, 0.60])
    for i, (lab, sub, e) in enumerate(EFFECTS):
        for s, c, dy, filled in [("all", T["grey"], -0.15, False), ("G3>0", T["blue"], 0.15, True)]:
            est, lo, hi, p = e[s]
            ax.hlines(i + dy, lo, hi, color=c, lw=3)
            ax.scatter(est, i + dy, s=200, color=c if filled else T["bg"], edgecolor=c, lw=2.5, zorder=3)
        fig.text(0.315, 0.0, "")
        ax.text(-2.55, i - 0.08, lab, ha="right", va="bottom", fontsize=19, weight="bold", color=T["ink"],
                transform=ax.transData, clip_on=False)
        ax.text(-2.55, i + 0.02, sub, ha="right", va="top", fontsize=14, color=T["ink2"], clip_on=False)
        ax.text(3.0, i, m(e["all"][0]), ha="left", va="center", fontsize=19, color=T["ink2"], clip_on=False)
        ax.text(3.75, i, m(e["G3>0"][0]), ha="left", va="center", fontsize=19, weight="bold", color=T["blue"],
                clip_on=False)
    ax.text(3.0, -0.6, "All", ha="left", fontsize=15, color=T["ink2"], clip_on=False, va="bottom")
    ax.text(3.75, -0.6, "Zeros\nremoved", ha="left", fontsize=15, color=T["blue"], weight="bold", clip_on=False,
            va="bottom", linespacing=1.0)
    ax.axvline(0, color=T["ink2"], lw=1.4)
    ax.set_ylim(len(EFFECTS) - 0.4, -0.75)
    ax.set_xlim(-2.4, 2.8); ax.set_yticks([])
    ax.set_xticks([-2, -1, 0, 1, 2], ["−2", "−1", "0", "+1", "+2"])
    ax.set_xlabel("Change in final grade (points out of 20)")
    xgrid(ax, T)
    return fig


for T in (LIGHT, DARK):
    save(chart08(T), "08_what_predicts_grades", T)

## 09 · Student A vs Student B (recomputed exactly as in notebook 03)

In [12]:
ACAD = "failures + log_abs + C(sex) + age + C(school) + C(address) + C(higher)"
FAMSOC = ("Medu + Fedu + goout + Dalc + Walc + freetime + health + C(internet) + C(schoolsup) + C(famsup)"
          " + C(paid) + C(romantic)")
CTRL = ACAD + " + " + FAMSOC
m3d = smf.ols("G3 ~ C(studytime) + " + CTRL, por).fit(cov_type="HC3")
typical = {}
for col in CTRL.replace("C(", "").replace(")", "").replace(" ", "").split("+"):
    col = "absences" if col == "log_abs" else col
    typical[col] = por[col].mode()[0] if por[col].dtype == object else por[col].median()
prof = pd.DataFrame({"Student A": dict(typical, studytime=4, absences=12, failures=1, schoolsup="no"),
                     "Student B": dict(typical, studytime=2, absences=2, failures=0, schoolsup="yes")}).T
prof["log_abs"] = np.log1p(prof.absences.astype(float))
for c_ in prof.columns:
    if c_ not in ["school", "sex", "address", "higher", "internet", "schoolsup", "famsup", "paid", "romantic"]:
        prof[c_] = prof[c_].astype(float)
sf = m3d.get_prediction(prof).summary_frame(alpha=0.05); sf.index = prof.index
chk = reg[(reg.section == "profile prediction") & (reg.model == "M3 + family/social") & (reg["sample"] == "all")]
assert np.allclose(sf["mean"].values, chk.estimate.values) and np.allclose(sf.obs_ci_lower.values, chk.pi_lo.values)
P = m3d.params
contrib = {"Study\n>10 vs\n2-5 h": P["C(studytime)[T.4]"] - P["C(studytime)[T.2]"],
           "Absences\n12 vs 2": P["log_abs"] * (np.log(13) - np.log(3)),
           "Failures\n1 vs 0": P["failures"],
           "Support\nno vs yes": -P["C(schoolsup)[T.yes]"]}
p2 = prof.copy(); p2["schoolsup"] = "no"
gap_equal = m3d.predict(p2).iloc[1] - m3d.predict(p2).iloc[0]     # B - A with support held equal
gap = sf["mean"].iloc[0] - sf["mean"].iloc[1]
assert np.isclose(sum(contrib.values()), gap)
print(sf.round(2)); print({k.replace(chr(10), ' '): round(v, 2) for k, v in contrib.items()},
                          "gap A-B", round(gap, 2), "| B leads with support equal by", round(gap_equal, 2))


def chart09(T):
    fig = canvas(T, "Student A “wins” only because Student B gets school support",
                 "A: >10 h study, 12 absences, 1 past failure, no support  ·  "
                 "B: 2-5 h, 2 absences, no failures, school support",
                 SRC + "  Model M3 (study-time dummies); other traits set to the typical student.")
    ax = fig.add_axes([0.085, 0.17, 0.42, 0.56])
    cols = {"Student A": T["blue"], "Student B": T["ink2"]}
    lo_ov, hi_ov = sf.obs_ci_lower.max(), sf.obs_ci_upper.min()
    ax.axvspan(lo_ov, hi_ov, color=T["band"], zorder=0)
    ax.text((lo_ov + hi_ov) / 2, 1.72, f"ranges overlap from {lo_ov:.1f} to {hi_ov:.1f}", ha="center", fontsize=15,
            color=T["ink2"])
    for i, (who, r) in enumerate(sf.iterrows()):
        c = cols[who]
        ax.hlines(i, r.obs_ci_lower, r.obs_ci_upper, color=c, lw=3, alpha=0.6)
        ax.hlines(i, r.mean_ci_lower, r.mean_ci_upper, color=c, lw=14)
        ax.scatter(r["mean"], i, s=320, color=T["bg"], edgecolor=c, lw=3.5, zorder=3)
        ax.text(r["mean"], i - 0.28, f"{r['mean']:.1f}", ha="center", va="bottom", fontsize=30, weight="bold")
        ax.text(-0.3, i, who.replace("Student ", ""), ha="right", va="center", fontsize=34, weight="bold", color=c)
    ax.set_ylim(1.85, -0.85); ax.set_xlim(0, 20); ax.set_yticks([])
    ax.set_xticks([0, 5, 10, 15, 20])
    ax.set_xlabel("Predicted final grade G3")
    xgrid(ax, T)
    ax.text(0, -0.9, "thick: 95% CI of the average  ·  thin: 95% range for one student",
            fontsize=14, color=T["ink2"], va="bottom")
    # waterfall
    axw = fig.add_axes([0.60, 0.17, 0.37, 0.56])
    run = 0.0
    labs = list(contrib) + ["A − B\ngap"]
    for k, (lab, v) in enumerate(contrib.items()):
        c = T["blue"] if v > 0 else T["ink2"]
        axw.bar(k, v, bottom=run, color=c, width=0.62)
        axw.text(k, run + v + (0.08 if v > 0 else -0.08), m(v), ha="center", va="bottom" if v > 0 else "top",
                 fontsize=18, weight="bold")
        if k < len(contrib) - 1:
            axw.plot([k + 0.31, k + 0.69], [run + v] * 2, color=T["muted"], lw=1.2)
        run += v
    axw.plot([len(contrib) - 1 + 0.31, len(contrib) - 0.31], [run] * 2, color=T["muted"], lw=1.2)
    axw.bar(len(contrib), run, color=T["ink"], width=0.62)
    axw.text(len(contrib), run + 0.08, m(run), ha="center", va="bottom", fontsize=18, weight="bold")
    axw.axhline(0, color=T["axis"], lw=1.4)
    axw.set_xticks(range(len(labs)), labs, fontsize=15)
    axw.set_ylabel("Points in A’s favour (+) or B’s (−)", fontsize=16)
    axw.set_ylim(-2.1, 2.1); axw.set_yticks([-2, -1, 0, 1, 2], ["−2", "−1", "0", "+1", "+2"])
    ygrid(axw, T)
    axw.set_title(f"Hold support equal and B leads by {gap_equal:.2f}", loc="left", fontsize=19, weight="bold",
                  color=T["ink"], pad=12)
    return fig


for T in (LIGHT, DARK):
    save(chart09(T), "09_student_a_vs_b", T)

            mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  \
Student A  12.60     0.55          11.52          13.68          7.25   
Student B  12.22     0.41          11.42          13.03          6.92   

           obs_ci_upper  
Student A         17.95  
Student B         17.52  
{'Study >10 vs 2-5 h': np.float64(0.7), 'Absences 12 vs 2': np.float64(-0.21), 'Failures 1 vs 0': np.float64(-1.42), 'Support no vs yes': np.float64(1.3)} gap A-B 0.37 | B leads with support equal by 0.93


## 10 · Diagnostics scorecard (from results/diagnostics_scorecard.csv)

In [13]:
CHECKS = [("Multicollinearity (max VIF)", "Multicollinearity (max VIF)", "VIF"),
          ("Heteroskedasticity, Breusch-Pagan (M3)", "Heteroskedasticity, Breusch-Pagan, M3", "p"),
          ("Heteroskedasticity, White (M3)", "Heteroskedasticity, White, M3", "p"),
          ("Heteroskedasticity, Breusch-Pagan (M4)", "Heteroskedasticity, Breusch-Pagan, M4", "p"),
          ("Normality of residuals, Jarque-Bera (M3)", "Normal residuals, Jarque-Bera, M3", "p"),
          ("Influence, Cook's distance (M3)", "Influential points, Cook's D, M3", "D"),
          ("Functional form, RESET power 2 (M3, log(abs+1))", "Functional form, RESET, M3", "p"),
          ("Autocorrelation (Durbin-Watson)", "Autocorrelation", "na")]
ACTION = {"Multicollinearity (max VIF)": "No action", "Heteroskedasticity, Breusch-Pagan (M3)": "HC3 robust SEs everywhere",
          "Heteroskedasticity, White (M3)": "HC3 robust SEs everywhere", "Heteroskedasticity, Breusch-Pagan (M4)": "HC3 robust SEs everywhere",
          "Normality of residuals, Jarque-Bera (M3)": "Large n; HC3 inference",
          "Influence, Cook's distance (M3)": "No single point dominates",
          "Functional form, RESET power 2 (M3, log(abs+1))": "Reported openly (0-20 scale)",
          "Autocorrelation (Durbin-Watson)": "Cross-sectional data"}


def cell(check, sample, kind):
    r = diag[(diag.check == check) & (diag["sample"] == sample)]
    if r.empty:
        r = diag[diag.check == check]
    r = r.iloc[0]
    light = r.light
    txt = {"VIF": f"max {r.statistic:.2f}", "p": fmt_p(r.p_value), "D": f"max {r.statistic:.3f}", "na": ""}[kind]
    return light, txt


def chart10(T):
    fig = canvas(T, "Dropping the 15 zero-grade cases turns most red lights green or amber",
                 "Regression diagnostics for the main models, all students vs zero-grade cases removed",
                 "Source: results/diagnostics_scorecard.csv (notebook 05). Pass / Watch / Fail as assigned there. "
                 "Max VIF is for the whole model; M3 = controls model, M4 = + G1, G2.")
    ax = fig.add_axes([0.045, 0.09, 0.93, 0.72]); ax.axis("off")
    ax.set_xlim(0, 100); ax.set_ylim(len(CHECKS) + 0.2, -0.9)
    cx = [0, 40, 60, 79]
    for x, h in zip(cx, ["Check", "All students", "Zeros removed", "What we do"]):
        ax.text(x, -0.55, h, fontsize=16, color=T["ink2"], weight="bold", va="center")
    ax.plot([0, 100], [-0.15, -0.15], color=T["axis"], lw=1.2)
    for i, (key, label, kind) in enumerate(CHECKS):
        y = i + 0.4
        ax.text(cx[0], y, label, fontsize=18, va="center", color=T["ink"])
        for col, sample in [(cx[1], "all"), (cx[2], "G3 > 0")]:
            light, txt = cell(key, sample, kind)
            col_hex, word, txtc = STATUS[light]
            if col_hex:
                ax.add_patch(FancyBboxPatch((col, y - 0.26), 6.4, 0.52, boxstyle="round,pad=0,rounding_size=0.22",
                                            fc=col_hex, ec="none", mutation_aspect=0.5))
                ax.text(col + 3.2, y, word, ha="center", va="center", fontsize=15, weight="bold", color=txtc)
            else:
                ax.text(col + 3.2, y, word, ha="center", va="center", fontsize=15, color=T["muted"])
            ax.text(col + 7.4, y, txt, va="center", fontsize=15, color=T["ink2"])
        ax.text(cx[3], y, ACTION[key], fontsize=16, va="center", color=T["ink2"])
        ax.plot([0, 100], [i + 0.9, i + 0.9], color=T["grid"], lw=1)
    return fig


for T in (LIGHT, DARK):
    save(chart10(T), "10_diagnostics_scorecard", T)

## 11 · Hero summary: three findings on one slide

In [14]:
def chart11(T):
    fig = canvas(T, "Effort shows up, history dominates, and absences matter once you check your zeros",
                 "Study Smarter, Not Harder  ·  649 Portuguese-course students, two schools, grades 0-20",
                 SRC + "  Models: OLS with HC3 SEs. Associations, not causes.")
    fig.texts[0].set_fontsize(26)
    zg = por[por.zero_grade]
    ba_a, ba_n = ba.loc["Portuguese, all"], ba.loc["Portuguese, G3 > 0"]
    cols = [(0.045, "1  EFFORT SHOWS UP", f"+{t13.value:.1f}", "points: <2 h vs 5-10 h of study a week,\nthen a plateau"),
            (0.375, "2  HISTORY DOMINATES", f"{lad_all.estimate.iloc[0]:.2f} {RA} {lad_all.estimate.iloc[3]:.2f}",
             "study-time effect once controls and\nearlier grades are added"),
            (0.705, "3  CHECK YOUR ZEROS", f"{int((zg.absences == 0).sum())} / {len(zg)}",
             "zero-grade cases had zero absences;\nwithout them absences cost points")]
    for k, (x0, kick, big, small) in enumerate(cols):
        fig.text(x0, 0.79, kick, fontsize=16, weight="bold", color=T["orange"] if k == 2 else T["blue"])
        fig.text(x0, 0.755, big, fontsize=50, weight="bold", color=T["ink"], va="top")
        fig.text(x0, 0.625, small, fontsize=17, color=T["ink2"], va="top", linespacing=1.3)
        ax = fig.add_axes([x0 + 0.01, 0.13, 0.25, 0.36])
        if k == 0:
            mu = grp["mean"].values
            ax.plot(range(4), mu, color=T["muted"], lw=1.5)
            ax.scatter(range(4), mu, s=160, color=T["blue"], edgecolor=T["bg"], lw=2, zorder=3)
            for i, v in enumerate(mu):
                ax.text(i, v + 0.35, f"{v:.1f}", ha="center", fontsize=15, color=T["ink"])
            ax.set_xticks(range(4), list(ST.values()), fontsize=14)
            ax.set_ylim(9.5, 14.4); ax.set_yticks([])
            ax.set_xlim(-0.4, 3.4)
            ax.text(-0.4, 9.65, "mean G3 (axis from 9.5)", fontsize=12, color=T["muted"])
        elif k == 1:
            b = lad_all.estimate.values
            ax.bar(range(4), b, color=T["blue"], width=0.6)
            for i, v in enumerate(b):
                ax.text(i, v + 0.03, m(v), ha="center", va="bottom", fontsize=15)
            ax.set_xticks(range(4), ["M1", "M2", "M3", "M4"], fontsize=14)
            ax.set_ylim(0, 1.2); ax.set_yticks([])
            ax.text(3.4, 1.12, f"R² {lad_all.r2.iloc[0]:.2f} {RA} {lad_all.r2.iloc[3]:.2f}", ha="right",
                    fontsize=14, color=T["ink2"])
        else:
            vals = [ba_a.M3_logabs, ba_n.M3_logabs]
            ax.bar([0, 1], vals, color=[T["grey"], T["blue"]], width=0.55)
            for i, v in enumerate(vals):
                ax.text(i, v - 0.02, m(v), ha="center", va="top", fontsize=15)
            ax.axhline(0, color=T["axis"], lw=1.2)
            ax.set_xticks([0, 1], ["with zeros", "zeros removed"], fontsize=14)
            ax.tick_params(axis="x", pad=-0)
            ax.xaxis.set_ticks_position("top")
            ax.set_ylim(-0.62, 0.05); ax.set_yticks([]); ax.spines["bottom"].set_visible(False)
            ax.text(1.4, -0.6, "log-absence effect, M3", ha="right", fontsize=12, color=T["muted"])
            ax.set_xlim(-0.5, 1.5)
        ax.spines["left"].set_visible(False)
    return fig


for T in (LIGHT, DARK):
    save(chart11(T), "11_hero_summary", T)

## Saved files

In [15]:
for n in SAVED:
    print(f"figures/showcase/{n}.png  (+ .svg, + dark/{n}.png)")

figures/showcase/01_effort_pays_then_plateaus.png  (+ .svg, + dark/01_effort_pays_then_plateaus.png)
figures/showcase/02_coefficient_ladder_melts.png  (+ .svg, + dark/02_coefficient_ladder_melts.png)
figures/showcase/03_spike_at_zero.png  (+ .svg, + dark/03_spike_at_zero.png)
figures/showcase/04_perfect_attendance_paradox.png  (+ .svg, + dark/04_perfect_attendance_paradox.png)
figures/showcase/05_zero_grade_trajectories.png  (+ .svg, + dark/05_zero_grade_trajectories.png)
figures/showcase/06_absence_effect_before_after.png  (+ .svg, + dark/06_absence_effect_before_after.png)
figures/showcase/07_maths_replication.png  (+ .svg, + dark/07_maths_replication.png)
figures/showcase/08_what_predicts_grades.png  (+ .svg, + dark/08_what_predicts_grades.png)
figures/showcase/09_student_a_vs_b.png  (+ .svg, + dark/09_student_a_vs_b.png)
figures/showcase/10_diagnostics_scorecard.png  (+ .svg, + dark/10_diagnostics_scorecard.png)
figures/showcase/11_hero_summary.png  (+ .svg, + dark/11_hero_summary.